# Rico - LoRA fine-tuning on Kaggle (free T4) | تدريب ريكو على كاجل

Same training as `ml/train_lora.py` on your PC, but on a free **Kaggle T4 (16 GB)**. Produces the Rico-lite GGUF shards
(Q4_K_M, <=1.9 GB each) + a catalog patch.

**Before you run / قبل التشغيل**
1. Notebook settings -> **Accelerator: GPU T4 x2** (or P100) and **Internet: On**.
2. *(Optional)* Add-ons -> Secrets: `HF_TOKEN` (faster HF downloads), `GH_TOKEN` (only if the repo is private or you want the upload cell).
3. Training data: either attach a Kaggle Dataset containing `rico_sft.jsonl` (named `rico-sft`), or leave it and the notebook uses `ml/data/rico_sft.jsonl` from the repo.
4. Run all cells. Expected: ~3-7 h for 2 epochs (checkpoint every 50 steps). Sessions last up to 12 h; the weekly GPU quota is ~30 h.
5. Disk: the merged 16-bit model, the bf16 GGUF and the Q4_K_M intermediate live in `/tmp` (not counted in the 20 GB output); only `release/` (shards + mmproj + catalog patch) and the small LoRA adapter stay in `/kaggle/working`.

**Resuming**: if the session dies, *Save Version -> Save & Run All (Commit)* keeps `/kaggle/working`. In the next session add that notebook's output as an
input and set `RESUME_FROM = "/kaggle/input/<your-notebook>/outputs"` in the parameters cell - checkpoints are copied back and training resumes.

**Base model**: `qwen35-4b` (Qwen3.5-4B, Apache-2.0 = catalog base of rico-lite). 16-bit LoRA in fp16 (T4 has no bf16).
If it fails or the loss is NaN, set `PRESET = "qwen3-4b-2507"` (plain-transformer fallback).

In [ ]:
!nvidia-smi -L
import sys, platform
print(sys.version)
print(platform.platform())

In [ ]:
# ---- parameters (edit) ----------------------------------------------------------------------------------
REPO_URL    = "https://github.com/giumaa/rico-ai"   # for a private repo the GH_TOKEN secret is used
BRANCH      = "main"
PRESET      = "qwen35-4b"        # qwen35-4b | qwen3-4b-2507 | qwen35-2b | qwen3-1.7b
EPOCHS      = 2
SEQ_LEN     = 1024
EXPORT      = True               # merge -> GGUF -> Q4_K_M -> Rico metadata -> shards
UPLOAD      = False              # upload shards to the GitHub release (needs GH_TOKEN secret with repo scope)
RESUME_FROM = ""                 # e.g. /kaggle/input/<prev-notebook>/outputs  (copied into /kaggle/working/outputs)
MAX_SAMPLES = 0                  # 0 = all; use e.g. 500 for a smoke test
RELEASE_TAG = "models-v2"        # release for the shards (models-v1 is FROZEN)
REV         = "ft1"              # asset names: rico-lite-<REV>-Q4_K_M-...  Bump for every new fine-tune (never overwritten)
TMP         = "/tmp/rico-work"   # merged/ + bf16 + Q4_K_M intermediates (kept out of /kaggle/working)

In [ ]:
import os, subprocess, shutil, glob


def sh(cmd, **kw):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, **kw)


os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single-GPU training
os.environ["TOKENIZERS_PARALLELISM"] = "false"
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    for k in ("HF_TOKEN", "GH_TOKEN"):
        try:
            os.environ[k] = sec.get_secret(k)
        except Exception:
            pass
except Exception:
    pass

# ---- install ----------------------------------------------------------------------------------------
sh("pip install -q --upgrade unsloth unsloth_zoo")
sh('pip install -q "transformers>=5.2,<=5.5.0" "peft>=0.18" accelerate bitsandbytes "datasets<4.4" gguf jinja2 safetensors sentencepiece')

# ---- repo -------------------------------------------------------------------------------------------
WORK = "/kaggle/working"
REPO = f"{WORK}/rico-ai"
if not os.path.isdir(REPO):
    url = REPO_URL
    if os.environ.get("GH_TOKEN"):
        url = url.replace("https://", f"https://x-access-token:{os.environ['GH_TOKEN']}@")
    sh(f"git clone --depth 1 --branch {BRANCH} {url} {REPO}")
sh(f"cd {REPO} && git log -1 --oneline")

In [ ]:
# ---- data + resume ----------------------------------------------------------------------------------
cands = glob.glob("/kaggle/input/**/rico_sft.jsonl", recursive=True)
DATA = cands[0] if cands else f"{REPO}/ml/data/rico_sft.jsonl"
assert os.path.exists(DATA), "rico_sft.jsonl not found - run the 'Distill' workflow first or attach it as a dataset"
print("training data:", DATA, sum(1 for _ in open(DATA, encoding="utf8")), "records")

OUT = f"{WORK}/outputs/rico-lite"
if RESUME_FROM and os.path.isdir(RESUME_FROM):
    sh(f"mkdir -p {WORK}/outputs && cp -r {RESUME_FROM}/* {WORK}/outputs/")
    print("copied checkpoints from", RESUME_FROM)

# sanity check: prints one example (prompt masked), runs 2 real train steps and a merge dry-run (adapter keys vs. a fresh base)
sh(f"cd {REPO} && python ml/train_lora.py --preset {PRESET} --data {DATA} --out {OUT} --dry-run")

In [ ]:
# ---- train (auto-resumes from OUT/checkpoint-* if present) -------------------------------------------
extra = f"--max-samples {MAX_SAMPLES}" if MAX_SAMPLES else ""
sh(f"cd {REPO} && python ml/train_lora.py --preset {PRESET} --data {DATA} --out {OUT} "
   f"--epochs {EPOCHS} --seq-len {SEQ_LEN} --save-steps 50 {extra}")

In [ ]:
# ---- export: merge LoRA -> GGUF -> Q4_K_M -> Rico metadata + identity template -> <=1.9 GB shards -----------
# merged/ (16-bit), the bf16 GGUF and the Q4_K_M file go to TMP (/tmp); only OUT/release is written to /kaggle/working
if EXPORT:
    sh(f"cd {REPO} && python ml/train_lora.py --preset {PRESET} --data {DATA} --out {OUT} --skip-train --export "
       f"--work-dir {TMP} --llama-work /tmp/llama.cpp --release-tag {RELEASE_TAG} --rev {REV}")
    sh(f"ls -la {OUT}/release")
    shutil.rmtree(TMP, ignore_errors=True)


In [ ]:
# ---- (optional) upload shards to the GitHub release RELEASE_TAG - NEVER overwrites an existing asset ---------------
# Shipped apps embed the sha256 of every asset they download, so a name is immutable. Same name + same size -> skipped,
# same name + different size -> hard error (bump REV).
if UPLOAD:
    import json, urllib.request
    owner_repo = REPO_URL.rstrip("/").split("github.com/")[1]
    tok = os.environ["GH_TOKEN"]
    hdr = {"Authorization": f"Bearer {tok}", "Accept": "application/vnd.github+json"}

    def api(url, method=None, data=None):
        req = urllib.request.Request(url, method=method, headers=hdr, data=data)
        with urllib.request.urlopen(req) as r:
            return json.loads(r.read())

    assert RELEASE_TAG != "models-v1", "models-v1 is frozen"
    try:
        rel = api(f"https://api.github.com/repos/{owner_repo}/releases/tags/{RELEASE_TAG}")
    except Exception:
        rel = api(f"https://api.github.com/repos/{owner_repo}/releases", method="POST",
                  data=json.dumps({"tag_name": RELEASE_TAG, "name": f"Rico models ({RELEASE_TAG})", "make_latest": "false",
                                   "body": "Model shards for the Rico app (see docs/ml.md)"}).encode())
    existing = {a["name"]: a["size"] for a in rel["assets"]}
    for f in sorted(glob.glob(f"{OUT}/release/*.gguf")):
        name, size = os.path.basename(f), os.path.getsize(f)
        if name in existing:
            if existing[name] != size:
                raise SystemExit(f"{name} already exists in {RELEASE_TAG} with a different size - refusing to overwrite; bump REV")
            print("skip (already uploaded):", name)
            continue
        up = rel["upload_url"].split("{")[0] + f"?name={name}"
        sh(f'curl -sS -f -X POST -H "Authorization: Bearer {tok}" -H "Content-Type: application/octet-stream" '
           f'--data-binary @{f} "{up}" -o /dev/null')
        print("uploaded", name)


In [ ]:
# ---- package results (Output tab) --------------------------------------------------------------------
sh(f"cd {OUT} && ls -la adapter release 2>/dev/null | head -50")
sh(f"cd {OUT} && tar czf {WORK}/rico-lite-adapter.tar.gz adapter")   # small LoRA adapter (+ rico_meta.json), reusable
# only release/ (shards + mmproj + catalog patch) and the adapter remain in /kaggle/working (20 GB output limit)
shutil.rmtree(TMP, ignore_errors=True)
for d in glob.glob(f"{OUT}/checkpoint-*") + [f"{OUT}/dryrun"]:
    shutil.rmtree(d, ignore_errors=True)
sh(f"du -sh {WORK}/* | sort -h | tail -8")


## After the run | بعد التشغيل
* Download `outputs/rico-lite/release/*.gguf` and `catalog-patch-rico-lite.json` from the **Output** tab (or use the upload cell, or the *Train on Kaggle* workflow with `action=fetch`).
* Release assets are **never overwritten**: names are `rico-lite-<REV>-Q4_K_M-0000N-of-0000M.gguf` (and `rico-lite-mmproj-<sha8>-F16.gguf`). The release `models-v1` is frozen - fine-tunes go to `models-v2`. For the next fine-tune bump `REV` (`ft2`, ...).
* On your PC (alternative to the workflow): `bash .github/scripts/upload_assets.sh models-v2 <the .gguf files>`, then
  `python ml/package_model.py merge-catalog --patch-dir <folder with catalog-patch-rico-lite.json>` and commit `models/catalog.json`.
* The fine-tuned catalog entry gets `finetuned: true`; its single-file HF fallback stays the *base* model (the app still injects the full Rico persona; the GGUF template only carries the short identity line).

Troubleshooting: *OOM* -> `SEQ_LEN = 768` or `PRESET = "qwen3-4b-2507"`. *NaN loss in fp16 with qwen35-4b* -> use `qwen3-4b-2507`.
*Unsloth import error* -> the script falls back to plain PEFT automatically (`--engine peft` forces it).
*Merge says adapter modules are missing in the base* -> the adapter and the export used different model classes; `adapter/rico_meta.json` records the loader used for training and the merge re-uses it.
